# Load Shedding by Bus vs Seasonal Storage Buses

This notebook checks whether load shedding occurs at buses that host seasonal storage.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display


In [ ]:
# --- User configuration -----------------------------------------------------
PROJECT_DIR = Path("/home/fs01/jl2966/acorn-julia2/acorn-julia")
RUN_NAME = "low_RE_mod_elec_iter0"
CLIMATE_SCENARIO = "historical_1980_2019"

RUNS = {
    "week_cap": "stagel_3month_1week_cap",
    "month_cap": "stagel_3month_1month_cap",
}

YEAR = 1985

output_root = PROJECT_DIR / "runs" / RUN_NAME / "outputs" / CLIMATE_SCENARIO
print("Runs:", RUNS)
print("Year:", YEAR)


In [ ]:
def load_shedding_by_bus(run_dir: Path, year: int) -> pd.DataFrame:
    df = pd.read_csv(run_dir / f"load_shedding_{year}.csv")
    # drop zone if present
    cols = [c for c in df.columns if c not in ("zone",)]
    df = df[cols]
    # melt to long
    value_cols = [c for c in df.columns if c != "bus_id"]
    long = df.melt(id_vars=["bus_id"], value_vars=value_cols,
                   var_name="timestamp", value_name="load_shed")
    long["load_shed"] = pd.to_numeric(long["load_shed"], errors="coerce").fillna(0.0)
    by_bus = long.groupby("bus_id", as_index=False)["load_shed"].sum()
    return by_bus


def load_seasonal_buses(run_dir: Path) -> set[int]:
    debug = pd.read_csv(run_dir / "_storage_debug.csv")
    if "is_seasonal" not in debug.columns:
        return set()
    seasonal = debug[debug["is_seasonal"] == 1]
    return set(seasonal["bus_id"].astype(int).tolist())


In [ ]:
# --- Compare shedding vs seasonal buses ------------------------------------
rows = []
for label, name in RUNS.items():
    run_dir = output_root / name
    shed = load_shedding_by_bus(run_dir, YEAR)
    seasonal_buses = load_seasonal_buses(run_dir)

    shed["is_seasonal_bus"] = shed["bus_id"].astype(int).isin(seasonal_buses)

    total_shed = shed["load_shed"].sum()
    shed_on_seasonal = shed.loc[shed["is_seasonal_bus"], "load_shed"].sum()
    share = (shed_on_seasonal / total_shed) if total_shed > 0 else 0.0

    rows.append({
        "run": label,
        "total_shed_MWh": total_shed,
        "shed_on_seasonal_MWh": shed_on_seasonal,
        "share_on_seasonal": share,
        "num_seasonal_buses": len(seasonal_buses),
    })

summary = pd.DataFrame(rows)
summary["share_on_seasonal"] = summary["share_on_seasonal"].round(4)
display(summary)


In [ ]:
# --- Top shedding buses + seasonal flag ------------------------------------
for label, name in RUNS.items():
    run_dir = output_root / name
    shed = load_shedding_by_bus(run_dir, YEAR)
    seasonal_buses = load_seasonal_buses(run_dir)
    shed["is_seasonal_bus"] = shed["bus_id"].astype(int).isin(seasonal_buses)

    print(f"
{label} – top 15 shedding buses")
    display(shed.sort_values("load_shed", ascending=False).head(15))
